In [2]:
import cv2
import csv
import time
import io
import math
import shutil
import struct
import subprocess
import sys
import threading
import wave
from pathlib import Path
import urllib.request
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# --------------------------------------------------
# Download model
# --------------------------------------------------

model_path = "hand_landmarker.task"

urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
    model_path
)


# --------------------------------------------------
# MediaPipe setup
# --------------------------------------------------

options = vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path=model_path
    ),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

landmarker = vision.HandLandmarker.create_from_options(options)


# --------------------------------------------------
# CSV setup
# --------------------------------------------------

csv_file = open("hand_data.csv", "w", newline="")
writer = csv.writer(csv_file)

header = [
    "timestamp_ms",
    "frame",
    "hand"
]

for i in range(21):
    header += [
        f"x{i}",
        f"y{i}",
        f"z{i}"
    ]

writer.writerow(header)


# --------------------------------------------------
# Countdown preview and optional sound
# --------------------------------------------------

COUNTDOWN_SECONDS = 3
SOUND_ENABLED = True  # Set False for a silent countdown.
WINDOW_NAME = "Hand Tracking"


def play_countdown_sound(go=False):
    """Play a short cue in the background when an audio player is available."""
    if not SOUND_ENABLED:
        return

    def play():
        frequency = 1000 if go else 600
        duration = 0.25 if go else 0.12
        try:
            if sys.platform == "win32":
                import winsound
                winsound.Beep(frequency, int(duration * 1000))
                return
            player = shutil.which("aplay") or shutil.which("paplay")
            if player is None:
                return  # The visual countdown works without sound support.
            sample_rate = 22050
            samples = b"".join(
                struct.pack("<h", int(8000 * math.sin(2 * math.pi * frequency * i / sample_rate)))
                for i in range(int(sample_rate * duration))
            )
            audio = io.BytesIO()
            with wave.open(audio, "wb") as wav:
                wav.setnchannels(1)
                wav.setsampwidth(2)
                wav.setframerate(sample_rate)
                wav.writeframes(samples)
            args = [player, "-q"] if Path(player).name == "aplay" else [player, "/dev/stdin"]
            subprocess.run(args, input=audio.getvalue(), stdout=subprocess.DEVNULL,
                           stderr=subprocess.DEVNULL, timeout=2)
        except (OSError, subprocess.SubprocessError):
            pass

    threading.Thread(target=play, daemon=True).start()


def draw_status(frame, message, active_light):
    """Draw a traffic light and readable status over the camera preview."""
    height, width = frame.shape[:2]
    panel_height = min(110, height)
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (width, panel_height), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.8, frame, 0.2, 0, frame)
    colors = [(0, 0, 255), (0, 190, 255), (0, 220, 0)]
    for index, color in enumerate(colors):
        cv2.circle(frame, (30 + index * 48, 30), 16,
                   color if index == active_light else (65, 65, 65), -1)
    scale = min(0.65, max(0.3, (width - 24) / 620))
    cv2.putText(frame, message, (12, 72), cv2.FONT_HERSHEY_SIMPLEX,
                scale, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(frame, "Q: stop | M: toggle sound", (12, 98),
                cv2.FONT_HERSHEY_SIMPLEX, scale * 0.8,
                (210, 210, 210), 1, cv2.LINE_AA)


# --------------------------------------------------
# Webcam: wait for Space, count down, then save data
# --------------------------------------------------

cap = cv2.VideoCapture(0)
frame_number = 0
countdown_start = None
start_time = None
last_count = None

try:
    if not cap.isOpened():
        raise RuntimeError("Could not open webcam")

    while True:
        success, frame = cap.read()
        if not success:
            break
        frame = cv2.flip(frame, 1)
        now = time.perf_counter()

        if countdown_start is None:
            draw_status(frame, "Press SPACE to start | Sound: " +
                        ("on" if SOUND_ENABLED else "off"), None)
        elif start_time is None:
            remaining = max(0, COUNTDOWN_SECONDS - int(now - countdown_start))
            if remaining != last_count:
                play_countdown_sound(go=remaining == 0)
                last_count = remaining
            if remaining > 0:
                draw_status(frame, f"Get ready... {remaining}",
                            0 if remaining > 1 else 1)
            else:
                start_time = now

        if start_time is not None:
            frame_number += 1
            rgb = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb
            )

            timestamp_ms = int(
                (now - start_time) * 1000
            )

            result = landmarker.detect_for_video(
                mp_image,
                timestamp_ms
            )


            # --------------------------------------------------
            # Save hand data
            # --------------------------------------------------

            for hand_index, hand in enumerate(
                result.hand_landmarks
            ):

                row = [
                    timestamp_ms,
                    frame_number,
                    hand_index
                ]

                for landmark in hand:
                    row += [
                        landmark.x,
                        landmark.y,
                        landmark.z
                    ]

                writer.writerow(row)


                # Draw landmarks
                h, w, _ = frame.shape

                for landmark in hand:
                    x = int(landmark.x * w)
                    y = int(landmark.y * h)

                    cv2.circle(
                        frame,
                        (x, y),
                        4,
                        (0, 255, 0),
                        -1
                    )


            draw_status(frame, "GO! Recording" if now - start_time < 1 else
                        f"Recording | {now - start_time:.1f}s", 2)

        cv2.imshow(WINDOW_NAME, frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q") or cv2.getWindowProperty(WINDOW_NAME, cv2.WND_PROP_VISIBLE) < 1:
            break
        if key == ord("m"):
            SOUND_ENABLED = not SOUND_ENABLED
        if key == ord(" ") and countdown_start is None:
            countdown_start = time.perf_counter()
finally:
    cap.release()
    landmarker.close()
    csv_file.close()
    cv2.destroyAllWindows()

print("Saved to hand_data.csv")


W0000 00:00:1791365489.578228   24119 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791365489.609505   24118 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved to hand_data.csv
